# XMCA-Net-Lite: 02 - Qwen2-VL RAG Answer Generation
### Purpose
Generate answers for all dataset splits using frozen `Qwen2-VL-2B-Instruct` loaded in 4-bit NF4 quantization.
Features:
- Shards capped at <= 250 items.
- Resumption tracked via `completed_ids.json`.
- Outputs saved as Parquet and JSON.


In [ ]:
import sys, json
from pathlib import Path
# Robust project root discovery (supports Kaggle uppercase/lowercase and local)
PROJECT_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/XMCA"), Path("/kaggle/working/xmca")]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.generation.qwen import Qwen2VLGenerator
from src.retrieval.indexer import DenseRetriever
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
# Load FAISS index for retrieved context passages
corpora_dir = Path(config["paths"]["corpora_dir"])
retriever = DenseRetriever()
retriever.load(corpora_dir / "faiss_index.bin", corpora_dir / "corpus_metadata.json")

# Load all split records
splits_dir = Path(config["paths"]["splits_dir"])
records_to_generate = []
for p in splits_dir.glob("*/*.json"):
    if not p.name.endswith("manifest.json"):
        with open(p, "r", encoding="utf-8") as f:
            records_to_generate.extend(json.load(f))

print(f"Total dataset items to process: {len(records_to_generate)}")

# Pre-retrieve top passages
evidence_map = {}
for item in records_to_generate:
    item_id = str(item["id"])
    top_docs = retriever.search(item["question"], top_k=config["models"]["retriever"]["top_k"])
    evidence_map[item_id] = [doc["text"] for doc, _ in top_docs]


In [ ]:
# Execute Qwen2-VL generation in <= 250 item shards with completed_ids.json checkpointing
gen_dir = Path(config["paths"]["generations_dir"]) / "qwen2vl"
generator = Qwen2VLGenerator(
    model_id=config["models"]["main_vlm"]["model_id"],
    quantization=config["models"]["main_vlm"]["quantization"],
    max_new_tokens=config["models"]["main_vlm"]["max_new_tokens"],
    temperature=config["models"]["main_vlm"]["temperature"]
)

generator.run_sharded_generation(
    records=records_to_generate,
    evidence_map=evidence_map,
    output_dir=gen_dir,
    shard_size=config["execution"]["shard_size"],
    prefix="qwen2vl_generation"
)

save_run_metadata(gen_dir, config, seed, "02_generate_qwen", {"total_items": len(records_to_generate)})
print("\nStep 02 completed. Qwen2-VL generation shards saved.")
